modificar las columnas del csv original porque vienen sucias y spark no puede leerlo

In [13]:
import os
import glob
import re

base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")
silver_dir = os.path.join(base_dir, "data", "silver")
os.makedirs(silver_dir, exist_ok=True)

target_years = ["2018", "2019", "2020", "2021"]

def clean_header_line(line):
    result = []
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and in_quotes:
            result.append('-')
        else:
            result.append(char)
    
    cleaned = ''.join(result)
    if cleaned.startswith('# '):
        cleaned = cleaned[2:]
    elif cleaned.startswith('#'):
        cleaned = cleaned[1:]
    
    columns = cleaned.split(',')
    clean_columns = []
    for col in columns:
        col_clean = re.sub(r'[^a-zA-Z0-9_ ]', '', col)
        col_clean = col_clean.replace(' ', '_')
        col_clean = re.sub(r'_+', '_', col_clean)
        col_clean = col_clean.strip('_')
        clean_columns.append(col_clean)
    
    return ','.join(clean_columns)

all_files = []
for year in target_years:
    pattern = os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", f"Turbine_Data_Kelmarsh_1_*.csv")
    all_files.extend(glob.glob(pattern))

print(f"📁 Archivos a procesar: {len(all_files)}")

for file_path in all_files:
    filename = os.path.basename(file_path)
    silver_path = os.path.join(silver_dir, f"cleaned_{filename}")
    
    print(f"\n🔄 Procesando: {filename}")
    
    with open(file_path, 'r', encoding='utf-8') as infile, \
         open(silver_path, 'w', encoding='utf-8') as outfile:
        
        line_number = 0
        for line in infile:
            line_number += 1
            
            if line_number < 10:
                continue
            
            if line_number == 10 and 'Date and time' in line:
                cleaned = clean_header_line(line)
                # AÑADIR \n AL FINAL PARA SEPARAR DEL SIGUIENTE CONTENIDO
                outfile.write(cleaned + '\n')
            else:
                outfile.write(line)
    
    print(f"   💾 Guardado en: {silver_path}")

print(f"\n✅ Todos los archivos procesados en: {silver_dir}")

📁 Archivos a procesar: 4

🔄 Procesando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv

✅ Todos los archivos procesados en: /home/aito

In [6]:
import os
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ==============================================================================
# 1. SPARK SESSION
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-EDA-Notebook") \
    .master("local[6]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# ==============================================================================
# 2. LECTURA DESDE CSV LIMPIOS
# ==============================================================================
base_dir = os.path.dirname(os.getcwd())
cleaned_dir = os.path.join(base_dir, "data", "silver")

target_years = ["2018", "2019", "2020", "2021"]
selected_paths = []
for year in target_years:
    pattern = os.path.join(cleaned_dir, f"cleaned_Turbine_Data_Kelmarsh_1_{year}*.csv")
    selected_paths.append(pattern)

all_files = []
for pattern in selected_paths:
    all_files.extend(glob.glob(pattern))

print(f"📁 Archivos encontrados: {len(all_files)}")
for f in all_files:
    print(f"   - {os.path.basename(f)}")

# ==============================================================================
# 3. CARGAR CON SPARK
# ==============================================================================
# Ahora el encabezado está en la línea 10 sin # inicial
# Las líneas 1-9 son comentarios (#), la 10 es el header válido
turbine_data_df = spark.read \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .csv(selected_paths)

# ==============================================================================
# 4. RESULTADOS
# ==============================================================================

print("\nVISTA PREVIA DE LAS PRIMERAS 5 FILAS:")
turbine_data_df.show(5, truncate=False)

total_records = turbine_data_df.count()
print(f"\n🎯 Total de registros: {total_records}")
print(f"📊 NÚMERO TOTAL DE COLUMNAS: {len(turbine_data_df.columns)}")

📁 Archivos encontrados: 4
   - cleaned_Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv



VISTA PREVIA DE LAS PRIMERAS 5 FILAS:
+-------------------+------------------+--------------------------------+---------------------+---------------------+-----------------+----------------------+-----------------------------------------+------------------------------+------------------------------+----------------------+-----------------------------------------+------------------------------+------------------------------+------------------------------+------------------+------------------+---------------------------------+----------------------+----------------------+-----------------------------------+------------------------+------------------------+-------------------+--------------------+--------------------+-----------------------+-----------------+-------------------------+-----------------+-------------------------+-------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-----------------

In [15]:
from pyspark.sql import functions as F

# ==============================================================================
# ANÁLISIS DE NULOS Y NaN POR COLUMNA
# ==============================================================================
print("=" * 80)
print("📊 ANÁLISIS DE VALORES NULOS Y NaN POR COLUMNA")
print("=" * 80)

# Crear expresiones de agregación para contar nulos + NaN de todas las columnas a la vez
agg_exprs = []
for i, col_name in enumerate(turbine_data_df.columns):
    # Detectar: isNull() OR (casteado a string == 'NaN')
    agg_exprs.append(
        F.sum(
            F.when(
                F.col(f"`{col_name}`").isNull() | (F.col(f"`{col_name}`").cast("string") == 'NaN'),
                1
            ).otherwise(0)
        ).alias(f"null_col_{i}")
    )

# Ejecutar una sola agregación
null_summary = turbine_data_df.agg(*agg_exprs).collect()[0]
total_rows = turbine_data_df.count()

# Procesar resultados
null_counts = []
for i, col_name in enumerate(turbine_data_df.columns):
    null_count = null_summary[f"null_col_{i}"]
    null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
    null_counts.append((col_name, null_count, null_pct))

# Ordenar por nulos descendente
null_counts_sorted = sorted(null_counts, key=lambda x: x[1], reverse=True)

print(f"\n📋 Total de filas: {total_rows}")
print(f"📋 Total de columnas: {len(turbine_data_df.columns)}")

cols_with_nulls = [(c, n, p) for c, n, p in null_counts_sorted if n > 0]
print(f"\n🔴 Columnas con nulos/NaN: {len(cols_with_nulls)} de {len(turbine_data_df.columns)}")

if cols_with_nulls:
    print("\n" + "-" * 80)
    print(f"{'Columna':<<50} {'Nulos+NaN':>10} {'% Total':>10}")
    print("-" * 80)
    for col_name, null_count, null_pct in cols_with_nulls:
        print(f"{col_name:<50} {null_count:>10} {null_pct:>9.2f}%")
    
    print("\n" + "-" * 80)
    print("TOP 20 COLUMNAS CON MÁS NULOS/NaN:")
    print("-" * 80)
    for col_name, null_count, null_pct in cols_with_nulls[:20]:
        bar = "█" * int(null_pct / 2)
        print(f"{col_name:<45} {null_count:>8} ({null_pct:>5.1f}%) {bar}")
else:
    print("\n✅ No hay columnas con valores nulos ni NaN")

# Columnas completamente vacías (100% nulos/NaN)
fully_null = [(c, n) for c, n, p in null_counts_sorted if p == 100]
if fully_null:
    print(f"\n⚠️  COLUMNAS COMPLETAMENTE VACÍAS (100% nulos/NaN): {len(fully_null)}")
    for col_name, _ in fully_null:
        print(f"   - {col_name}")

# Columnas sin nulos ni NaN
cols_no_nulls = [c for c, n, p in null_counts if n == 0]
print(f"\n🟢 Columnas sin nulos ni NaN: {len(cols_no_nulls)}")

📊 ANÁLISIS DE VALORES NULOS Y NaN POR COLUMNA


26/05/30 17:38:34 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 303, schema size: 299
CSV file: file:///home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv



📋 Total de filas: 210384
📋 Total de columnas: 299

🔴 Columnas con nulos/NaN: 295 de 299

--------------------------------------------------------------------------------
Columna<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<  Nulos+NaN    % Total
--------------------------------------------------------------------------------
Potential_power_met_mast_anemometer_kW                 210384    100.00%
Potential_power_met_mast_anemometer_MPC_kW             210384    100.00%
Potential_power_estimated_kW                           207517     98.64%
Energy_Import_counter_kWh                              187635     89.19%
Equivalent_Full_Load_Hours_counter_s                   158341     75.26%
Lost_Production_Contractual_Custom_kWh                 158334     75.26%
Productionbased_Contractual_Avail_Custom               158334     75.26%
Lost_Production_Contractual_Global_kWh                 157828     75.02%
Timebased_Contractual_Avail_Custom                     157828     75.02%
Productionbased_Co